##Project Goal:
1. Baseline Model: lyric text only
2. controlled model: same lyric text, but control tokens added
3. Research Question: Does adding control tokens help the model generate lyrics that better match the requested style?


In [ ]:
#imports
#!pip install transformers datasets evaluate accelerate #hugging face import

import pandas as pd
import numpy as np
import math
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, TrainingArguments, Trainer)


##Data Loading & Cleaning
data found on github:https://github.com/walkerkq/musiclyrics/blob/master/billboard_lyrics_1964-2015.csv



In [ ]:
import pandas as pd
import re

# load and inspect
df = pd.read_csv("/content/billboard_lyrics_1964-2015.csv", encoding="latin1")

# make columns lowercase once and for all
df.columns = df.columns.str.lower()

print(df.columns)
print(df.shape)
display(df.head())

# drop rows with missing/empty lyrics
df = df.dropna(subset=["lyrics"]).copy()
df["lyrics"] = df["lyrics"].astype(str).str.strip()
df = df[df["lyrics"] != ""]

# remove placeholder rows
bad_phrases = ["instrumental", "lyrics unavailable", "no lyrics"]
df = df[~df["lyrics"].str.lower().isin(bad_phrases)]

# standardize text fields
def clean_basic(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["song"] = df["song"].apply(clean_basic)
df["artist"] = df["artist"].apply(clean_basic)

# create cleaned artist column for duplicate matching
def normalize_artist(text):
    text = text.lower()
    text = re.sub(r"\b(feat\.?|featuring|with|and)\b.*", "", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["artist_clean"] = df["artist"].apply(normalize_artist)

# clean lyric text
def clean_lyrics(text):
    text = str(text)

    # remove bracketed section labels like [Verse], [Chorus]
    text = re.sub(r"\[.*?\]", "", text)

    # remove simple repeat markers / parenthetical metadata
    text = re.sub(r"\((x\d+|repeat.*?)\)", "", text, flags=re.IGNORECASE)

    # normalize curly quotes/apostrophes
    text = text.replace("’", "'").replace("“", '"').replace("”", '"')

    # collapse spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # collapse repeated blank lines
    text = re.sub(r"\n\s*\n+", "\n", text)

    return text.strip()

df["lyrics_clean"] = df["lyrics"].apply(clean_lyrics)

# remove duplicates
def normalize_lyrics_for_match(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\n]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["lyrics_norm"] = df["lyrics_clean"].apply(normalize_lyrics_for_match)

df = df.drop_duplicates(subset=["song", "artist_clean", "year"])
df = df.drop_duplicates(subset=["song", "artist_clean", "lyrics_norm"])

# filter out unusable songs
df["word_count"] = df["lyrics_clean"].str.split().str.len()
df = df[(df["word_count"] >= 40) & (df["word_count"] <= 500)].copy()

print("After cleaning/filtering:", df.shape)

# split each song lyric into smaller chunks
rows = []

for _, row in df.iterrows():
    chunks = [c.strip() for c in row["lyrics_clean"].split("\n") if c.strip()]

    for i in range(0, len(chunks), 4):
        chunk = "\n".join(chunks[i:i+4]).strip()

        if len(chunk.split()) >= 20:
            rows.append({
                "song": row["song"],
                "artist": row["artist"],
                "year": row["year"],
                "rank": row["rank"],
                "source": row["source"],
                "lyrics_chunk": chunk
            })

chunk_df = pd.DataFrame(rows)

print("Chunked dataset shape:", chunk_df.shape)
display(chunk_df.head())

def assign_decade(year):
    year = int(year)
    if 1965 <= year <= 1969:
        return "1960s"
    elif 1970 <= year <= 1979:
        return "1970s"
    elif 1980 <= year <= 1989:
        return "1980s"
    elif 1990 <= year <= 1999:
        return "1990s"
    elif 2000 <= year <= 2009:
        return "2000s"
    elif 2010 <= year <= 2015:
        return "2010s"
    else:
        return None

chunk_df["decade"] = chunk_df["year"].apply(assign_decade)

print(chunk_df["decade"].value_counts())
display(chunk_df.head())

#save as file
chunk_df.to_csv("/content/cleaned_lyrics_chunks.csv", index=False)

##Data Clean-up Continued.
I cleaned up the csv file manually in google sheets and inputted the mood based on the lyric-chunk.


In [ ]:
import re

df = pd.read_csv("/content/lyrics_final.csv")

#fix encoding artifacts
def fix_encoding_text(text):
    text = str(text)
    text = text.replace("Ì¢", "'")
    text = text.replace("Ã¢", "'")
    text = text.replace("Ã", "")
    text = text.replace("�", "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["lyrics_chunk"] = df["lyrics_chunk"].apply(fix_encoding_text)

print(df.columns)
print(df.shape)
df.head()

df.to_csv("/content/lyrics_model_ready.csv", index=False)

##Split train, validation, test by song


In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv("/content/lyrics_model_ready.csv")

#split by each unique song
songs = df["song"].drop_duplicates()

train_songs, temp_songs = train_test_split(songs, test_size=0.3, random_state=42)
val_songs, test_songs = train_test_split(temp_songs, test_size=0.5, random_state=42)

train_df = df[df["song"].isin(train_songs)].copy()
val_df = df[df["song"].isin(val_songs)].copy()
test_df = df[df["song"].isin(test_songs)].copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

# create model text formats for ALL splits
for split_df in [train_df, val_df, test_df]:
    split_df["baseline_text"] = split_df["lyrics_chunk"]

    split_df["controlled_text"] = (
        "<MOOD=" + split_df["mood"].astype(str) + "> "
        + "<DECADE=" + split_df["decade"].astype(str) + ">\n"
        + split_df["lyrics_chunk"].astype(str)
    )

# now save AFTER creating the text columns
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/val.csv", index=False)
test_df.to_csv("/content/test.csv", index=False)

# quick check
display(train_df[["mood", "decade", "baseline_text", "controlled_text"]].head(3))

In [ ]:
print(set(train_df["song"]).intersection(set(val_df["song"])))
print(set(train_df["song"]).intersection(set(test_df["song"])))
print(set(val_df["song"]).intersection(set(test_df["song"])))

##Training the baseline model using DistilGPT-2

In [ ]:
#imports
from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          DataCollatorForLanguageModeling,
                          TrainingArguments,
                          Trainer)

In [ ]:
#load the split files
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

print(train_df.columns)
train_df.head()



In [ ]:
#baseline datasets
baseline_train = Dataset.from_pandas(train_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))
baseline_val = Dataset.from_pandas(val_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))
baseline_test = Dataset.from_pandas(test_df[["baseline_text"]].rename(columns={"baseline_text": "text"}))

In [ ]:
#load distilgpt-2 tokenizer and load model
model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
#tokenize text
def tokenize_fn (examples):
  return tokenizer(examples["text"])

tokenized_train = baseline_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val = baseline_val.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_test = baseline_test.map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
#grouping into chunks
block_size = 128

def group_texts(examples):
  concatenated = {k: sum(examples[k], []) for k in examples.keys()}
  total_length = len(concatenated["input_ids"])
  total_length = (total_length // block_size) * block_size
  result = {
      k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
      for k , t in concatenated.items()
  }
  result["labels"] = result["input_ids"].copy()
  return result

lm_train = tokenized_train.map(group_texts, batched = True)
lm_val = tokenized_val.map(group_texts, batched = True)
lm_test = tokenized_test.map(group_texts, batched = True)

In [ ]:
#collate data
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
#train setup
training_args = TrainingArguments(
    output_dir="/content/baseline_distilgpt2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=20,
    fp16=True,
    report_to="none"
)
#train baseline model
trainer = Trainer(
    model=model,
    args = training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
    processing_class = tokenizer,
)
trainer.train()

In [ ]:
#eval baseline
eval_results = trainer.evaluate()
print(eval_results)

baseline_perplexity = math.exp(eval_results["eval_loss"])
print("Baseline perplexity:", baseline_perplexity)

In [ ]:
#save baseline
trainer.save_model("/content/baseline_distilgpt2/final")
tokenizer.save_pretrained("/content/baseline_distilgpt2/final")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_baseline = AutoTokenizer.from_pretrained("/content/baseline_distilgpt2/final")
model_baseline = AutoModelForCausalLM.from_pretrained("/content/baseline_distilgpt2/final").to(device)
model_baseline.eval()

In [ ]:
#generate baseline samples
prompt = "write short song lyrics\n"
inputs = tokenizer_baseline(prompt, return_tensors="pt").to(device)

output = model_baseline.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_k=40,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
)

print(tokenizer_baseline.decode(output[0], skip_special_tokens=True))

In [ ]:
#save baseline sample results
baseline_eval = trainer.evaluate()
baseline_perplexity = math.exp(baseline_eval["eval_loss"])

print("Baseline eval loss:", baseline_eval["eval_loss"])
print("Baseline perplexity:", baseline_perplexity)

##Building the Controlled Datasets Using DistilGPT-2



In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

controlled_train = Dataset.from_pandas(
    train_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)
controlled_val = Dataset.from_pandas(
    val_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)
controlled_test = Dataset.from_pandas(
    test_df[["controlled_text"]].rename(columns={"controlled_text": "text"})
)

In [ ]:
#reload distilgpt
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
#inserting my control tokens
special_tokens = {
    "additional_special_tokens": [
        "<MOOD=happy>",
        "<MOOD=sad>",
        "<MOOD=reflective>",
        "<MOOD=confident>",
        "<DECADE=1960s>",
        "<DECADE=1970s>",
        "<DECADE=1980s>",
        "<DECADE=1990s>",
        "<DECADE=2000s>",
        "<DECADE=2010s>",
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)
model.resize_token_embeddings(len(tokenizer))

print("Added tokens:", num_added)

In [ ]:
#tokenize the controlled text
def tokenize_fn(examples):
    return tokenizer(examples["text"])

tokenized_train = controlled_train.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val   = controlled_val.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_test  = controlled_test.map(tokenize_fn, batched=True, remove_columns=["text"])

In [ ]:
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    total_length = (total_length // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_train = tokenized_train.map(group_texts, batched=True)
lm_val   = tokenized_val.map(group_texts, batched=True)
lm_test  = tokenized_test.map(group_texts, batched=True)

In [ ]:
#data collator
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from transformers import TrainingArguments
#set training arguments
training_args = TrainingArguments(
    output_dir="/content/controlled_distilgpt2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=20,
    fp16=True,
    report_to="none"
)

In [ ]:
#training the controlled model
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
#evaluate controlled model
import math

controlled_eval = trainer.evaluate()
controlled_perplexity = math.exp(controlled_eval["eval_loss"])

print("Controlled eval loss:", controlled_eval["eval_loss"])
print("Controlled perplexity:", controlled_perplexity)

In [ ]:
#save controlled model
trainer.save_model("/content/controlled_distilgpt2/final")
tokenizer.save_pretrained("/content/controlled_distilgpt2/final")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_controlled = AutoTokenizer.from_pretrained("/content/controlled_distilgpt2/final")
model_controlled = AutoModelForCausalLM.from_pretrained("/content/controlled_distilgpt2/final").to(device)
model_controlled.eval()

In [ ]:
# controlled model generation
prompt = "<MOOD=happy> <DECADE=2010s>\n"
inputs = tokenizer_controlled(prompt, return_tensors="pt").to(device)

output = model_controlled.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_k=40,
    top_p=0.9,
    temperature=0.8,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
)

print(tokenizer_controlled.decode(output[0], skip_special_tokens=False))

In [ ]:
model_controlled.eval()

prompts = [
    "<MOOD=happy> <DECADE=1980s>\n",
    "<MOOD=sad> <DECADE=1990s>\n",
    "<MOOD=reflective> <DECADE=1970s>\n",
    "<MOOD=confident> <DECADE=2000s>\n",
]

controlled_samples = []

for prompt in prompts:
    inputs = tokenizer_controlled(prompt, return_tensors="pt").to(device)

    output = model_controlled.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    text = tokenizer_controlled.decode(output[0], skip_special_tokens=False)
    controlled_samples.append({"prompt": prompt, "generated_text": text})
    print("=" * 80)
    print(text)

In [ ]:
import pandas as pd

controlled_samples_df = pd.DataFrame(controlled_samples)
controlled_samples_df.to_csv("/content/controlled_generations.csv", index=False)
controlled_samples_df.head()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_baseline = AutoTokenizer.from_pretrained("/content/baseline_distilgpt2/final")
model_baseline = AutoModelForCausalLM.from_pretrained("/content/baseline_distilgpt2/final").to(device)


model_baseline.eval()

In [ ]:
baseline_samples = []

for i in range(4):
    prompt = "write short song lyrics\n"
    inputs = tokenizer_baseline(prompt, return_tensors="pt").to(model_baseline.device)

    output = model_baseline.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    text = tokenizer_baseline.decode(output[0], skip_special_tokens=True)
    baseline_samples.append({"prompt": prompt, "generated_text": text})

baseline_samples_df = pd.DataFrame(baseline_samples)
baseline_samples_df.to_csv("/content/baseline_generations.csv", index=False)

In [ ]:
results_df = pd.DataFrame([
    {"model": "baseline", "eval_loss": baseline_eval["eval_loss"], "perplexity": baseline_perplexity},
    {"model": "controlled", "eval_loss": controlled_eval["eval_loss"], "perplexity": controlled_perplexity},
])

results_df.to_csv("/content/model_metrics.csv", index=False)
results_df

In [ ]:
import pandas as pd

baseline_df = pd.read_csv("/content/baseline_generations.csv")
controlled_df = pd.read_csv("/content/controlled_generations.csv")

n = min(len(baseline_df), len(controlled_df))

comparison_df = pd.DataFrame({
    "baseline_prompt": baseline_df.iloc[:n]["prompt"].values,
    "baseline_output": baseline_df.iloc[:n]["generated_text"].values,
    "controlled_prompt": controlled_df.iloc[:n]["prompt"].values,
    "controlled_output": controlled_df.iloc[:n]["generated_text"].values,
})

comparison_df.to_csv("/content/comparison_outputs.csv", index=False)
comparison_df.head()

In [ ]:
human_eval_df = pd.DataFrame({
    "prompt": comparison_df["controlled_prompt"],
    "baseline_output": comparison_df["baseline_output"],
    "controlled_output": comparison_df["controlled_output"],
    "baseline_coherence": "",
    "controlled_coherence": "",
    "baseline_style_match": "",
    "controlled_style_match": "",
    "notes": ""
})

human_eval_df.to_csv("/content/human_eval_template.csv", index=False)
human_eval_df.head()

In [ ]:
import os
import shutil

project_folder = "/content/final_project_submission"
os.makedirs(project_folder, exist_ok=True)

files_to_copy = [
    "/content/baseline_generations.csv",
    "/content/controlled_generations.csv",
    "/content/comparison_outputs.csv",
    "/content/human_eval_template.csv",
    "/content/lyrics_final.csv",
    "/content/lyrics_model_ready.csv",
    "/content/model_metrics.csv",
    "/content/test.csv",
    "/content/train.csv",
    "/content/val.csv",
]

for file_path in files_to_copy:
    if os.path.exists(file_path):
        shutil.copy(file_path, project_folder)

shutil.make_archive("/content/final_project_submission", "zip", project_folder)